# Priority Queue Contracts and Heap Shape


```{contents}
:local:
:depth: 2
```


```{index} Priority Describes Removal, Not a Fully Sorted Array
```



```{index} Priority Describes Removal, Not a Fully Sorted Array
```

## Priority Describes Removal, Not a Fully Sorted Array



A service desk chooses the lowest numbered priority next. That is a removal contract. It does not require that every stored element be maintained in complete sorted order, nor does it say that equal priorities must leave in arrival order. State the priority direction and tie policy before interpreting a printed storage array.

A min-heap requires each parent's priority to be no greater than its children's. The root is therefore a minimum, but siblings need not be ordered. In the array [1, 4, 2, 7, 6, 3], the values 4 and 2 can appear in that order without violating the heap. Reading the array from left to right is not the same as repeatedly removing the minimum.

| Operation needed | Heap property helps with it? | Reason |
| --- | --- | --- |
| Inspect the minimum | Yes | It is at the root |
| Remove the minimum and restore the property | Yes | Only a repair path is needed |
| Print everything in sorted order without changing storage | Not directly | Heap order is weaker than total order |
| Find an arbitrary ID | Not directly | Priority ancestry does not locate arbitrary keys |

The complete-tree shape lets an array encode parent/child positions without one node object per edge. With zero-based positions, a non-root index i has parent (i-1)/2 using integer division; its potential children are 2i+1 and 2i+2. A child position must still be checked against Count. Capacity is not the number of live heap entries.

Guided drawing: place the six array values above into the implicit tree. Check each parent-child relation. Then compare positions one and two: their values are out of sorted-array order but are allowed as siblings. This single counterexample explains why ordinary binary search cannot be applied to heap storage.

For related explanation, see the further reading below.[^development-reading]


```{index} Priority Queue Behavior
```

## Priority Queue Behavior




```{index} priority queue
```


A **priority queue** removes the item with the next priority, not necessarily the item that arrived first. In C#'s `PriorityQueue<TElement, TPriority>`, lower priority values come out first by default.


In [1]:
using System;
using System.Collections.Generic;

var tasks = new PriorityQueue<string, int>();
tasks.Enqueue("Process invoice", 2);
tasks.Enqueue("Respond to outage", 0);
tasks.Enqueue("Send summary", 3);
tasks.Enqueue("Reset password", 1);

while (tasks.Count > 0)
{
    string task = tasks.Dequeue();
    Console.WriteLine(task);
}


Respond to outage
Reset password
Process invoice
Send summary


The outage has priority `0`, so it is processed before lower-urgency work. Priority queues are useful for schedulers, event simulations, pathfinding, triage, and any workflow where "next" means most important rather than oldest.


**Common mistake.** Expecting `foreach (var item in tasks)` to list the work in priority order. Enumeration follows the internal heap layout, which is not sorted; only repeated `Dequeue` calls deliver items by priority.


A priority queue defines how you select the next element. It does not promise that enumeration is sorted or that arbitrary keys are easy to locate. The numerical priority has an application meaning: here zero is most urgent. Dequeue from an empty queue is an error; `TryDequeue` reports false instead.

Equal priorities need a policy. The .NET library does not guarantee arrival order for ties. If the service desk requires first-arrived service among equally urgent tickets, use the pair `(urgency, arrivalNumber)` as the priority. Tuple comparison considers urgency first and arrival second. Each arrival number is unique within the queue, so the resulting order is deterministic.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

var queue = new PriorityQueue<string, (int Urgency, long Arrival)>();
long arrival = 0;
queue.Enqueue("T-100", (2, arrival++));
queue.Enqueue("T-101", (0, arrival++));
queue.Enqueue("T-102", (2, arrival++));
while (queue.TryDequeue(out string? ticket, out var priority))
    Console.WriteLine($"{ticket}: urgency={priority.Urgency}, arrival={priority.Arrival}");
Console.WriteLine($"empty={queue.Count == 0}");

T-101: urgency=0, arrival=1
T-100: urgency=2, arrival=0
T-102: urgency=2, arrival=2
empty=True


The order is T-101, T-100, T-102. A later urgent ticket can move ahead, while equal-urgency tickets retain their arrival order. The queue alone does not prevent starvation if urgent work arrives indefinitely. Fairness requires an additional scheduling policy.

The manual implementation in this section is a **binary** heap. The current .NET `PriorityQueue` uses four children per parent internally. Its API behavior is the same priority-queue idea, but the binary index formulas below describe the teaching implementation, not the library's internal layout. [Microsoft's PriorityQueue documentation](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.priorityqueue-2) describes both tie behavior and the library representation.

```{index} Why Not a Sorted List?
```

### Why Not a Sorted List?

Before accepting a new structure, check that the familiar ones really fall short. Each candidate below is asked to support the two priority-queue operations, add an item and remove the most urgent one, for n items.

| Representation | Add | Remove most urgent | Weakness |
| --- | --- | --- | --- |
| Unsorted list or array | O(1) append | O(n) scan for the minimum, then remove | Every removal reads everything |
| Sorted array | O(n) shift to keep order | O(1) at the end | Every insertion moves half the items |
| Sorted linked list | O(n) walk to the position | O(1) at the head | Same walk, no indexing |
| Binary search tree | O(h) | O(h) to find and remove the minimum | h is O(log n) only if the tree stays balanced |
| Binary heap | O(log n) | O(log n) | None for these two operations; no fast search by key |

The lists make one operation cheap by making the other linear. The BST makes both logarithmic on average but offers no guarantee against the chain of Chapter 17. The heap gives a guaranteed logarithmic bound for both, and it does so by promising less than a BST. A BST maintains a *total* order: for any two nodes, the inorder position tells which key is smaller. A heap maintains a *partial* order: the relation is known only between a node and its descendants. Two siblings, or two nodes in different subtrees, may be in either order. That weaker promise is exactly enough to keep the minimum at the root, and it is cheaper to maintain because an update repairs one path instead of an entire ordering.

```{index} Heap Representation
```

## Heap Representation




```{index} heap; min-heap; complete tree
```


A **heap** is a common priority-queue representation. A binary heap has two key rules:

- **Shape rule**: the tree is complete, filled level by level from left to right.
- **Order rule**: in a min-heap, each parent priority is less than or equal to its child priorities.

Because the shape is complete, a heap can be stored compactly in an array. For an item at index `i`, the children are at `2 * i + 1` and `2 * i + 2`.


In [1]:
using System;

int[] heap = { 1, 3, 2, 7, 5, 6, 4 };

for (int parent = 0; parent < heap.Length; parent++)
{
    int left = 2 * parent + 1;
    int right = 2 * parent + 2;

    if (left < heap.Length)
    {
        Console.WriteLine($"{heap[parent]} -> left child {heap[left]}");
    }

    if (right < heap.Length)
    {
        Console.WriteLine($"{heap[parent]} -> right child {heap[right]}");
    }
}


1 -> left child 3
1 -> right child 2
3 -> left child 7
3 -> right child 5
2 -> left child 6
2 -> right child 4


The array is not fully sorted. It only guarantees that each parent is no larger than its children. That weaker rule is enough to keep the minimum at the root.


**Common mistake.** Checking the heap rule only at the root, or comparing siblings. The order rule relates each parent to its own children: in `1, 3, 2, 7, 5, 6, 4` the siblings 3 and 2 are out of numeric order and the heap is still valid, while a single parent larger than one of its children would invalidate it.


For a non-root index i, its parent is `(i - 1) / 2`, using integer division. Valid child indexes must be less than Count. Count describes occupied positions; the backing array may have extra capacity that is not part of the heap.

```text
          1 (0)
        /       \
     3 (1)     2 (2)
    /    \     /    \
 7 (3) 5 (4) 6 (5) 4 (6)
```

The values obey parent-child order, but siblings and separate subtrees can be out of numerical order. For example, 3 precedes 2 in the array. Repeatedly removing the root produces a sorted sequence; reading the raw array does not. The complete shape guarantees height Θ(log n) for a nonempty binary heap.

The same arithmetic answers two more questions that implementations need. The parent of index i is (i - 1) / 2 with integer division, so the formula also finds a node's ancestors by repeated application, which is the path that bubble-up follows. And because the tree is complete, the internal nodes occupy exactly the first half of the array: for n items, indexes 0 through n / 2 - 1 are internal and indexes n / 2 through n - 1 are leaves. That boundary matters in the next section, where bottom-up heap construction starts at the last internal node and leaves need no work. The example prints each index's relatives and marks the leaf boundary for a heap of seven and of six items.

In [ ]:
using System;

foreach (int n in new[] { 7, 6 })
{
    Console.WriteLine($"n = {n}: internal indexes 0..{n / 2 - 1}, leaf indexes {n / 2}..{n - 1}");
    for (int i = 0; i < n; i++)
    {
        string parent = i == 0 ? "none" : ((i - 1) / 2).ToString();
        int left = 2 * i + 1, right = 2 * i + 2;
        string children = left >= n ? "none (leaf)" : right >= n ? $"{left} only" : $"{left} and {right}";
        Console.WriteLine($"  index {i}: parent {parent}, children {children}");
    }
}

n = 7: internal indexes 0..2, leaf indexes 3..6
  index 0: parent none, children 1 and 2
  index 1: parent 0, children 3 and 4
  index 2: parent 0, children 5 and 6
  index 3: parent 1, children none (leaf)
  index 4: parent 1, children none (leaf)
  index 5: parent 2, children none (leaf)
  index 6: parent 2, children none (leaf)
n = 6: internal indexes 0..2, leaf indexes 3..5
  index 0: parent none, children 1 and 2
  index 1: parent 0, children 3 and 4
  index 2: parent 0, children 5 only
  index 3: parent 1, children none (leaf)
  index 4: parent 1, children none (leaf)
  index 5: parent 2, children none (leaf)


For seven items the three internal nodes are indexes 0, 1, and 2, and each has two children; for six items the internal nodes are still 0, 1, and 2, but index 2 has only a left child, index 5. In both cases the first leaf is at index n / 2, which is why a loop that repairs internal nodes can stop there. Note that index 3's parent is index 1 whether n is six or seven; the parent formula depends only on i, while the child formulas must be checked against n because a computed child index can lie beyond the live items.

```{index} Code Tracing: Is This Array a Heap?
```

### Code Tracing: Is This Array a Heap?

Each array below is proposed as a min-heap. Draw the implicit tree for each, check every parent-child pair, and decide whether the array is a valid min-heap. If it is not, name the first pair, by parent index, that breaks the rule.

1. `[2, 6, 3, 8, 7, 5]`
2. `[2, 6, 3, 8, 7, 1]`
3. `[4, 4, 4, 9]`
4. `[1, 9, 2, 10, 12, 3, 7, 14, 11]`

Then predict the output of the solution, which prints the verdict and the first violating pair for each.

In [ ]:
// Solution
using System;

int[][] candidates =
{
    new[] { 2, 6, 3, 8, 7, 5 },
    new[] { 2, 6, 3, 8, 7, 1 },
    new[] { 4, 4, 4, 9 },
    new[] { 1, 9, 2, 10, 12, 3, 7, 14, 11 },
};
foreach (int[] a in candidates)
{
    string verdict = "valid min-heap";
    for (int child = 1; child < a.Length; child++)
    {
        int parent = (child - 1) / 2;
        if (a[parent] > a[child]) { verdict = $"invalid: index {parent} holds {a[parent]} above index {child} holding {a[child]}"; break; }
    }
    Console.WriteLine($"[{string.Join(", ", a)}] -> {verdict}");
}

[2, 6, 3, 8, 7, 5] -> valid min-heap
[2, 6, 3, 8, 7, 1] -> invalid: index 2 holds 3 above index 5 holding 1
[4, 4, 4, 9] -> valid min-heap
[1, 9, 2, 10, 12, 3, 7, 14, 11] -> valid min-heap


The first array is valid even though 6 sits before 3: they are siblings, and the heap rule says nothing about siblings. The second differs only in its last value, and 1 under parent 3 at index 2 breaks the rule. The third is valid because the rule allows equality; a heap can hold duplicates. The fourth is valid and shows the point of the partial order: the left subtree under 9 holds 10, 12, 14, and 11, all larger than everything in the right subtree under 2, which no BST would allow. The check visits every parent-child pair once, so validation is Θ(n), unlike the O(log n) operations the heap exists for.

A priority queue is an access rule. The next section implements it with a binary heap, while application code may use the library PriorityQueue. Enumeration is not a sorted report; repeated removal is the operation whose priority order is specified.


```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. A teammate prints a heap's array and says it is broken because the values are not ascending. What is the correct response?

```{dropdown} Answer
Ascending order is a BST-style total order and is not required. The heap promises only that each parent is no greater than its children. Check that rule pair by pair; if it holds, the heap is valid and repeated removal will still produce ascending output.
```

2. Why does a complete tree make an array representation possible, when an arbitrary binary tree would not?

```{dropdown} Answer
Completeness means the nodes fill levels from the top and from the left with no gaps, so the k-th node in level order is always at index k and the index formulas for parent and children hold everywhere. A tree with gaps would need placeholders for the missing nodes or explicit references.
```

3. A heap holds 20 items. Which indexes are leaves, and what is the index of the last node that has any children?

```{dropdown} Answer
Indexes 10 through 19 are leaves, since n / 2 is 10. The last internal node is index 9, whose left child is 19 and whose right child, index 20, does not exist.
```

4. Two tickets have equal urgency. Under the library's `PriorityQueue<TElement, TPriority>`, which comes out first, and how do you make it the one that arrived first?

```{dropdown} Answer
The library does not promise either order for ties. Make the priority a pair of urgency and an arrival counter; tuples compare the first component and then the second, so equal urgencies are broken by arrival.
```

```{index} Practice: Service the Minimum Priority
```

## Practice: Service the Minimum Priority




```{index} Practice: Service the Minimum Priority
```


Complete the operation, run the cell, and explain why the chosen item is returned.


In [ ]:
using System;
using System.Collections.Generic;

PriorityQueue<string, int> queue = new PriorityQueue<string, int>();
queue.Enqueue("Routine", 5); queue.Enqueue("Urgent", 1);
// TODO: explain the result.
Console.WriteLine(queue.Dequeue());


In [1]:
using System;
using System.Collections.Generic;

PriorityQueue<string, int> queue = new PriorityQueue<string, int>();
queue.Enqueue("Routine", 5); queue.Enqueue("Urgent", 1);
Console.WriteLine(queue.Dequeue());


Urgent


The output is `Urgent`. Its priority 1 is lower than Routine's 5, so it leaves first even though Routine was enqueued first; arrival order does not matter here. Swap the two priorities and `Routine` comes out instead.


```{rubric} Footnotes
```


[^1801_priority_queues-1]: Equal library priorities do not promise first-in-first-out ties. Supply a secondary arrival number when that is a requirement.


[^development-reading]: Further reading: [OpenDSA CS3, Heaps and Priority Queues](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/Heaps.html). The worked business scenarios and guided reasoning here are original teaching examples.
